## Command

---

> **In one line.** A command reifies an *action* as a first-class object $C = (\text{execute},\ \text{undo})$ — a forward operation paired with its exact inverse — so that calls become data: they can be queued, logged, replayed, and reversed through a $\text{History}$ stack, while the *invoker* that triggers $C$ stays fully decoupled from the *receiver* that performs it.

### 1. An action made into an object

Ordinarily an action is a transient event: you call a method on some object and it happens. The Command pattern **reifies** that event — it turns the call into a standing object $C$ that you can hold, store, and pass around. Each such command bundles three things inside itself: the **receiver** (the object that actually does the work — an editor, a light, a thermostat), the **forward operation** $\text{execute}$, and the **inverse operation** $\text{undo}$. We write the command as the pair

$$C = (\text{execute},\ \text{undo}).$$

Around the command sit two roles that never touch each other. The **invoker** triggers commands: it knows only the abstract command interface $C$ — how to call $\text{execute}$ and $\text{undo}$ — and never references the receiver directly. The **receiver** is encapsulated *inside* $C$. The invoker therefore depends on commands, not on the concrete objects those commands drive.

### 2. The round-trip law

The forward and inverse operations are not independent: $\text{undo}$ must reverse *exactly* what $\text{execute}$ did, returning the system to the state it left. Letting $\text{id}$ denote the identity function on the state, this is the **round-trip identity**, the central law of the pattern:

$$\boxed{\,C = (\text{execute},\ \text{undo}) \quad\text{with}\quad \text{undo} \circ \text{execute} = \text{id}\,}$$

Read pointwise on a state $\sigma$: running the action and then reversing it lands you exactly where you began,

$$\underbrace{\sigma}_{\text{before}} \;\xrightarrow{\;\text{execute}\;}\; \underbrace{\text{execute}(\sigma)}_{\text{after the action}} \;\xrightarrow{\;\text{undo}\;}\; \underbrace{\text{undo}(\text{execute}(\sigma))}_{=\ \sigma} .$$

This is why a `TypeCommand` must remember *how many* characters it added: only then can its $\text{undo}$ strip exactly those characters and satisfy $\text{undo} \circ \text{execute} = \text{id}$.

### 3. The History stack

Because commands are objects, executed ones can be **recorded in order**. The $\text{History}$ is an ordered list of the commands run so far,

$$\text{History} = [\,C_1, C_2, \ldots, C_n\,].$$

It is what makes undo and redo mechanical. Popping the last command $C_n$ and calling its $\text{undo}$ reverses the most recent action; replaying the sequence $[\,C_1, \ldots, C_n\,]$ in order reconstructs any state from the start. The whole edit history is now *data* you can walk forwards or backwards.

### 4. Data flow: invoker → command → receiver

The invoker issues an abstract call; the command translates it into concrete work on the receiver it hides:

$$\text{invoker} \;\xrightarrow{\;\text{press / execute}\;}\; C \;\xrightarrow{\;\text{does the work}\;}\; \text{receiver}.$$

Nothing flows back across the dependency: the invoker never learns which receiver $C$ holds, and the receiver never learns it is being driven through a command. The command object is the only thing that knows both.

### 5. Key conditions

1. **Round-trip identity** — the inverse undoes the forward operation exactly:
   $$\text{undo} \circ \text{execute} = \text{id}.$$
   Applying $\text{execute}$ then $\text{undo}$ returns the state to precisely where it was.
2. **Reification** — actions are first-class objects. Because a command is *data* rather than a bare function call, it can be queued, logged, serialised, or replayed.
3. **History enables undo/redo** — popping $C_n$ from $\text{History}$ and calling its $\text{undo}$ reverses the last action, and replaying $[\,C_1, \ldots, C_n\,]$ reconstructs any state.

&nbsp;

> 🧾 A restaurant order slip. The waiter writes the order ($C$) and hands it to the kitchen (the invoker). The slip encapsulates *what to do* — it can be queued, retried, or cancelled — and the waiter and kitchen stay fully decoupled, communicating only through the slip.

### Exercise 05 — Text Editor with Undo

---

**Scenario:** A text editor supports typing. Every action must be undoable. After several commands in History $= [C_1, C_2, C_3]$, pressing undo pops $C_3$ and calls its $\text{undo}$.

**Your task:** Implement `TypeCommand` with `execute()` and `undo()`, and an `Editor` maintaining the History stack.

```python
editor = Editor()
editor.execute(TypeCommand(editor, "Hello"))   # C_1
editor.execute(TypeCommand(editor, " World"))  # C_2
editor.undo()                                  # undo(C_2) — undo∘execute = id
print(editor.text)                             # Hello
```

**Hints**

- The editor holds `self._history = []`. `execute(cmd)` calls `cmd.execute()` then appends to History. `undo()` pops the last command and calls its `undo()`.
- `TypeCommand.undo()` removes exactly the characters that `execute()` added — store `self._len` so you know how many to strip. This enforces $\text{undo} \circ \text{execute} = \text{id}$.

In [ ]:
# --------------------------------
# Receiver: the Editor also acts as invoker, holding the History stack

class Editor:
    def __init__(self):
        self.text = ""
        self._history = []                   # History = [C_1, C_2, ..., C_n]

    def execute(self, cmd):                  # run forward op, then record in History
        # call cmd.execute(); append cmd to self._history
        ...

    def undo(self):                          # pop C_n, call its undo()  -> undo ∘ execute = id
        # if history is non-empty: pop last command, call its undo()
        ...

# --------------------------------
# Command: C = (execute, undo), encapsulating the receiver

class TypeCommand:
    def __init__(self, editor, chars):
        self._editor = editor                # the receiver, encapsulated inside C
        self._chars = chars
        self._len = len(chars)               # remember how much to strip on undo

    def execute(self):                       # forward operation: append chars
        ...

    def undo(self):                          # inverse: remove exactly self._len chars
        ...

# --------------------------------
editor = Editor()
editor.execute(TypeCommand(editor, "Hello"))    # C_1
editor.execute(TypeCommand(editor, " World"))   # C_2
editor.undo()                                   # undo(C_2)
print(editor.text)                              # expect: Hello

### Exercise 06 — Smart Home Remote

---

**Scenario:** A remote control (invoker) has programmable buttons — each mapped to a command $C$. The remote supports undo of the last action. It never imports `Light` or `Thermostat` directly.

**Your task:** Build a `RemoteControl` invoker and at least three command classes. The remote stores one last-command for undo.

```python
remote = RemoteControl()
remote.set_slot("A", LightOnCommand(light))
remote.press("A")    # execute C, store as last command
remote.undo()        # undo(C) — undo∘execute = id
```

**Hints**

- The remote holds `self._slots = {}` mapping button names to commands $C$. `press(slot)` calls `self._slots[slot].execute()` and stores it as the last command.
- The invoker knows only the command interface $C$ (`execute`/`undo`) — never the receiver. Each command encapsulates its own receiver and its inverse.

In [ ]:
# --------------------------------
# Receivers — the objects that actually perform the work

class Light:
    def __init__(self):
        self.on = False
    def turn_on(self):
        self.on = True
        print("Light: ON")
    def turn_off(self):
        self.on = False
        print("Light: OFF")

class Thermostat:
    def __init__(self):
        self.temp = 20
    def set_temp(self, t):
        self.temp = t
        print(f"Thermostat: {t}C")

# --------------------------------
# Commands — C = (execute, undo), each encapsulating its receiver

class LightOnCommand:
    def __init__(self, light):
        self._light = light
    def execute(self):                       # forward op
        ...
    def undo(self):                          # inverse: turn it back off
        ...

class LightOffCommand:
    def __init__(self, light):
        self._light = light
    def execute(self):
        ...
    def undo(self):                          # inverse: turn it back on
        ...

class SetTempCommand:
    def __init__(self, thermostat, temp):
        self._thermostat = thermostat
        self._temp = temp
        self._prev = None                    # remember old temp to restore on undo
    def execute(self):                       # store prev, then set new temp
        ...
    def undo(self):                          # inverse: restore self._prev
        ...

# --------------------------------
# Invoker — knows only the command interface C, never the receivers

class RemoteControl:
    def __init__(self):
        self._slots = {}                     # button name -> command C
        self._last = None                    # last executed command, for undo

    def set_slot(self, name, command):
        # map button name to its command C
        ...

    def press(self, name):                   # execute C, store as last command
        ...

    def undo(self):                          # undo the last command -> undo ∘ execute = id
        ...

# --------------------------------
light = Light()
thermostat = Thermostat()

remote = RemoteControl()
remote.set_slot("A", LightOnCommand(light))
remote.set_slot("B", LightOffCommand(light))
remote.set_slot("C", SetTempCommand(thermostat, 24))

remote.press("A")     # Light: ON
remote.undo()         # Light: OFF (undo of last)
remote.press("C")     # Thermostat: 24C
remote.undo()         # Thermostat: 20C (restored)